# Biotech FDA Catalyst Screening, SEC Cash Runway & rNPV Valuation Quickstart

This notebook demonstrates how to use the open-source **[BioSniper Biotech Toolkit](https://biosniper.co?utm_source=colab&utm_medium=open_source&utm_campaign=notebook_quickstart)** to:
1. Audit **SEC Form 10-Q/10-K unrestricted cash runway** against upcoming FDA PDUFA and Phase 2/3 clinical readout dates;
2. Run **Risk-Adjusted Net Present Value (rNPV)** valuation with a 3×3 WACC × Probability-of-Success (PoS) sensitivity matrix;
3. Query **ClinicalTrials.gov REST API v2** for primary endpoint definitions and study timelines;
4. Load the daily **[Biotech Catalyst & Cash Runway Benchmark Dataset](https://biosniper.co/calendar/pdufa?utm_source=colab&utm_medium=open_source&utm_campaign=notebook_quickstart)**.

In [1]:
import sys
from pathlib import Path
import json

# Add parent directory if running locally from notebooks/
if Path('../biosniper_toolkit.py').exists():
    sys.path.insert(0, str(Path('..').resolve()))

from biosniper_toolkit import (
    calculate_cash_runway,
    calculate_rnpv,
    load_biosniper_dataset,
)

## 1. Audit SEC Form 10-Q Cash Runway vs. Binary Catalyst Window

Clinical-stage biotech companies typically aim to maintain **6 to 12 months of cash buffer** beyond their next pivotal readout or PDUFA target action date to avoid forced dilutive financing ahead of the binary event.

You can also verify any ticker interactively on the **[BioSniper Cash Runway Calculator](https://biosniper.co/tools/cash-runway?utm_source=colab&utm_medium=open_source&utm_campaign=notebook_quickstart)**.

In [2]:
runway = calculate_cash_runway(
    cash_usd=185_000_000,
    quarterly_burn_usd=24_000_000,
    next_catalyst_months=8.0,
    ticker='RCKT',
    utm_source='colab',
)

print('Runway (months):', runway.runway_months)
print('Post-Catalyst Buffer (months):', runway.catalyst_buffer_months)
print('Dilution Warning (<12m):', runway.has_dilution_warning)
print('Interactive Simulator:', runway.interactive_calculator_url)
print('Company Evidence Page:', runway.company_evidence_url)

Runway (months): 23.1
Post-Catalyst Buffer (months): 15.1
Dilution Warning (<12m): False
Interactive Simulator: https://biosniper.co/tools/cash-runway?cash=185000000&burn=24000000&utm_source=colab&utm_medium=open_source&utm_campaign=runway_toolkit&utm_content=RCKT
Company Evidence Page: https://biosniper.co/company/RCKT?utm_source=colab&utm_medium=open_source&utm_campaign=runway_toolkit&utm_content=company_evidence


## 2. Risk-Adjusted NPV (rNPV) Valuation & Sensitivity Matrix

Reference methodology: **[Biotech Valuation: Risk-Adjusted NPV (rNPV) Explained](https://biosniper.co/blog/biotech-valuation-rnpv-explained?utm_source=colab&utm_medium=open_source&utm_campaign=notebook_quickstart)**.

In [3]:
val = calculate_rnpv(
    peak_sales_millions=1200.0,
    phase='Phase 3',
    therapeutic_area='rare_disease',
    remaining_rd_cost_millions=65.0,
    net_cash_millions=185.0,
    ticker='RCKT',
    utm_source='colab',
)

print(f"Phase 3 Rare Disease PoS: {val.pos_probability * 100:.1f}%")
print(f"Unadjusted NPV: ${val.unadjusted_npv_millions}M")
print(f"Risk-Adjusted Asset rNPV: ${val.rnpv_millions}M")
print(f"Enterprise rNPV (incl. Net Cash): ${val.enterprise_rnpv_millions}M")
print('Sensitivity Grid ($M):')
print(json.dumps(val.sensitivity_grid_millions, indent=2))

Phase 3 Rare Disease PoS: 65.7%
Unadjusted NPV: $1945.28M
Risk-Adjusted Asset rNPV: $1266.22M
Enterprise rNPV (incl. Net Cash): $1451.22M
Sensitivity Grid ($M):
{
  "WACC_10pct": {
    "PoS_minus_5pct": 1370.1,
    "PoS_base": 1485.08,
    "PoS_plus_5pct": 1600.06
  },
  "WACC_12pct": {
    "PoS_minus_5pct": 1168.18,
    "PoS_base": 1266.22,
    "PoS_plus_5pct": 1364.26
  },
  "WACC_14pct": {
    "PoS_minus_5pct": 1001.14,
    "PoS_base": 1085.15,
    "PoS_plus_5pct": 1169.17
  }
}


## 3. Query BioSniper's Verified PDUFA & SEC Cash-Runway Dataset

Reference hub: **[BioSniper PDUFA Calendar & Cash Runway Leaderboard](https://biosniper.co/calendar/pdufa?utm_source=colab&utm_medium=open_source&utm_campaign=notebook_quickstart)**.

In [4]:
pdufa_rows = load_biosniper_dataset(catalyst_type='PDUFA')
for r in pdufa_rows[:5]:
    print(r['ticker'], '|', r['expected_date'], '|', r['drug_name'] or r['indication'], '->', r['company_url'])

runway_watch = load_biosniper_dataset(max_runway_months=12.0)
for r in runway_watch[:5]:
    print(r['ticker'], '| Runway:', r['runway_months'], 'months ->', r['runway_calculator_url'])